# Use Ragas to evaluate the customized RAG pipeline based on milvus

**请注意，此测试需要消耗大量 LLM token。请仔细阅读并留意您请求访问的次数。**

## 1. 准备环境和数据

开始之前，您必须在环境变量中设置 DEEPSEEK_API_KEY（评估 LLM 使用 deepseek-v4-flash，
密钥存放在项目根目录的 .env 文件中）。

您还需要安装并启动 Milvus，可参考官方入门指南快速启动。

安装 pip 依赖项

In [1]:
# 必须在 import 任何会触发 huggingface_hub 的库之前设置缓存路径！
# （huggingface_hub 在导入时就把 HF_HOME/HF_HUB_CACHE 固化为默认的 C 盘路径）
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

# 缓存路径设置好之后，再进行下面的 import
import json
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
from beir import util


In [2]:
def prepare_fiqa_without_answer(knowledge_path):
    dataset_name="fiqa"

    if not os.path.exists(os.path.join(knowledge_path, f"{dataset_name}.zip")):
        url=("https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset_name))
        util.download_and_unzip(url,knowledge_path)

    data_path=os.path.join(knowledge_path, 'fiqa')
    with open(os.path.join(data_path, "corpus.jsonl")) as f:
        cs=[pd.Series(json.loads(l)) for l in f.readlines()]

    corpus_df=pd.DataFrame(cs)

    corpus_df=corpus_df.rename(columns={"_id":"corpus-id","text":"ground_truth"})
    corpus_df=corpus_df.drop(columns=["title","metadata"])
    corpus_df["corpus-id"]=corpus_df["corpus-id"].astype(int)
    corpus_df.head()

    with open(os.path.join(data_path, "queries.jsonl")) as f:
        qs=[pd.Series(json.loads(l)) for l in f.readlines()]

    queries_df=pd.DataFrame(qs)
    queries_df = queries_df.rename(columns={"_id": "query-id", "text": "question"})
    queries_df=queries_df.drop(columns=["metadata"])
    queries_df["query-id"]=queries_df["query-id"].astype(int)
    queries_df.head()

    splits=["dev", "test", "train"]
    split_df={}
    for s in splits:
        split_df[s]=pd.read_csv(os.path.join(data_path, f"qrels/{s}.tsv"),sep="\t").drop(columns=["score"])

    final_split_df={}
    for split in split_df:
        df=queries_df.merge(split_df[split], on="query-id")
        df=df.merge(corpus_df, on="corpus-id")
        df=df.drop(columns=["corpus-id"])
        grouped=df.groupby("query-id").apply(
            lambda x:pd.Series(
                {
                    "question": x["question"].sample().values[0],
                    "ground_truth": x["ground_truth"].tolist(),
                }
            )
        )
        grouped=grouped.reset_index()
        grouped=grouped.drop(columns=["query-id"])
        final_split_df[split]=grouped

    return final_split_df

In [3]:
knowledge_datas_path='./knowledge_datas'
fiqa_path=os.path.join(knowledge_datas_path, 'fiqa_doc.txt')

if not os.path.exists(knowledge_datas_path):
    os.mkdir(knowledge_datas_path)
contexts_list=[]
answer_list=[]

final_split_df=prepare_fiqa_without_answer(knowledge_datas_path)

docs=[]

split="test"
for ds in final_split_df[split]["ground_truth"]:
    docs.extend([d for d in ds])
print(len(docs))

docs_str='\n'.join(docs)
with open(fiqa_path,"w",encoding="utf-8") as f:
    f.write(docs_str)

split='test'
question_list=final_split_df[split]["question"].to_list()
ground_truth_list=final_split_df[split]["ground_truth"].to_list()

1706


现在我们有了问题列表和正确答案列表，知识文档已准备在 fiqa_path 中。

## 2. Build RAG pipeline based on milvus and langchain

使用 langchain 的 RecursiveCharacterTextSplitter 对文档进行分割。

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=TextLoader(fiqa_path,encoding="utf-8")
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=40)
docs=text_splitter.split_documents(documents)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_23112\284524186.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_milvus import Milvus

# 使用本地下载好的模型目录（F 盘，避免每次从 HF Hub 下载到 C 盘）
EMB_MODEL_PATH = r'F:\Teewon\Milvue\models\bge-base-en'
embeddings=HuggingFaceEmbeddings(model_name=EMB_MODEL_PATH)
vector_db=Milvus.from_documents(
    docs,
    embeddings,
    connection_args={"host":"127.0.0.1","port":19530},
    drop_old=True,
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
# RAG 回答生成：检索 Milvus + deepseek-v4-flash 生成回答（替代原版的 OpenAI agent）
from langchain_deepseek import ChatDeepSeek

# 从项目根目录 .env 加载 DEEPSEEK_API_KEY
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

def search_milvus(question, top_k=5):
    return vector_db.similarity_search(question, k=top_k)

chat_llm = ChatDeepSeek(model="deepseek-v4-flash", temperature=0.3)

def answer_with_rag(question):
    """检索 top-k 上下文，并用 deepseek-v4-flash 生成回答。返回 (answer, contexts)。"""
    docs = search_milvus(question)
    contexts = [d.page_content for d in docs]
    context_block = "\n\n".join(contexts)
    prompt = (
        "You are a financial QA assistant. Answer the question using ONLY the "
        "context below. If the context does not contain the answer, say you don't know.\n\n"
        f"Context:\n{context_block}\n\nQuestion: {question}\nAnswer:"
    )
    resp = chat_llm.invoke(prompt)
    return resp.content, contexts


In [9]:
# ---- vertexai shim：langchain-community 0.4.2 移除了 chat_models.vertexai，
# 而 ragas 0.4.x 仍会 import ChatVertexAI（仅用于 isinstance 检查，不会被实例化）。
# 在 import ragas 之前注入 stub 模块，避免改动已安装的包。
import sys, types

_vmod = types.ModuleType('langchain_community.chat_models.vertexai')
class ChatVertexAI:  # placeholder
    pass
_vmod.ChatVertexAI = ChatVertexAI
sys.modules['langchain_community.chat_models.vertexai'] = _vmod

import langchain_community.llms as _ll
if not hasattr(_ll, 'VertexAI'):
    class VertexAI:  # placeholder
        pass
    _ll.VertexAI = VertexAI

# ---- 用 deepseek-v4-flash 做 ragas 评估 LLM（替代原版默认 OpenAI）----
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

ragas_llm = ChatDeepSeek(model="deepseek-v4-flash", temperature=1e-8)

import os as _os
print('DEEPSEEK_API_KEY loaded:', bool(_os.environ.get('DEEPSEEK_API_KEY')))
print('ragas version:', __import__('ragas').__version__)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_23112\965845515.py:20: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
C:\Users\Administrator\AppData\Local\Temp\ipykernel_23112\965845515.py:20: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


DEEPSEEK_API_KEY loaded: True
ragas version: 0.4.3


In [11]:
# 用测试集的前 N 个问题做评估（原版是全部 test 集，非常耗时耗 token）
N = 5
sample_questions = question_list[:N]
sample_truths = [gt[0] if isinstance(gt, list) else gt for gt in ground_truth_list[:N]]

contexts_list = []
answer_list = []
for q in tqdm(sample_questions):
    ans, ctxs = answer_with_rag(q)
    answer_list.append(ans)
    contexts_list.append(ctxs)

ds = Dataset.from_dict({
    "question": sample_questions,
    "contexts": contexts_list,
    "answer": answer_list,
    "ground_truth": sample_truths,
})

result = evaluate(
    ds,
    metrics=[context_precision, context_recall],
    llm=ragas_llm,
    embeddings=embeddings,
)
print(result)


100%|██████████| 5/5 [00:13<00:00,  2.60s/it]


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'context_precision': 0.6400, 'context_recall': 0.6000}
